# Ppto27 · 00 Setup
Crea las carpetas del Volume y las tablas de control. Se corre **una sola vez** (o cuando cambie la estructura).

```
/Volumes/catalogo_ppto/esquema_ppto/vol_administrado/ppto27/
├── 00_codigo/                 ← ppto27_lib.py, ppto27_silver.py, ppto27_gold.py, ppto27_spark.py
├── 10_cargas/                 ← cada carga en su carpeta AAAAMMDD_HHMM, con las carpetas tal cual SharePoint ("en frío")
│   └── 20261015_0900/
│       ├── 02. Gastos de Personal/
│       ├── 03. Gastos Administrativos/
│       ├── 04. Tecnología y Comunicaciones/
│       ├── 05. Depreciación y Amortización/
│       └── 06. Impuestos y contribuciones/
└── 90_maestros/               ← maestros_ppto27.xlsx, alias_ppto27.xlsx y drivers_ppto27.xlsx
```
La versión que se envió a las áreas se sube una vez como carga `REF_AAAAMMDD` (sirve para los flags C: cambios en el Real).
Una carga puede traer todo o solo las plantillas que cambiaron: el pipeline se queda con la última versión de cada plantilla.

In [ ]:
%pip install openpyxl --quiet

In [ ]:
VOLUME = '/Volumes/catalogo_ppto/esquema_ppto/vol_administrado/ppto27'
for carpeta in ('00_codigo', '10_cargas', '90_maestros'):
    dbutils.fs.mkdirs(f'{VOLUME}/{carpeta}')
    print('OK', f'{VOLUME}/{carpeta}')

import os, sys
sys.path.append(f'{VOLUME}/00_codigo')
faltan = [f for f in ('ppto27_lib.py', 'ppto27_silver.py', 'ppto27_gold.py', 'ppto27_spark.py') if not os.path.exists(f'{VOLUME}/00_codigo/{f}')]
if faltan:
    raise Exception(f'Carpetas creadas. Ahora sube a ppto27/00_codigo: {faltan} y vuelve a correr este notebook')
import ppto27_lib as L

## Tablas de control
* `ppto27_ctl_cargas`: una fila por carga procesada.
* `ppto27_brz_archivos`: una fila por archivo leído en cada carga (hash, identificación, estado). Es el historial de versiones.
* `ppto27_brz_lineas`: filas de Tabla1 tal cual (texto), con la carga y el archivo de origen.

In [ ]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {L.tabla('ctl_cargas')} (
  carga_id STRING, ruta STRING, fecha_proceso TIMESTAMP, usuario STRING,
  archivos INT, archivos_nuevos INT, archivos_repetidos INT, archivos_rechazados INT, lineas INT)
USING DELTA COMMENT 'Ppto27: cargas procesadas (una por carpeta de 10_cargas)'
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {L.tabla('brz_archivos')} (
  carga_id STRING, ruta_relativa STRING, archivo STRING, sha256 STRING, tamano_bytes BIGINT, modificado TIMESTAMP,
  id_plantilla STRING, rubro_cod STRING, rubro STRING, sub_rubro STRING, tipo STRING, area STRING, pais_carpeta STRING, nombre_base STRING,
  estado STRING, error STRING, repetido_de STRING, hojas STRING, tablas_oculta BOOLEAN, ref_tabla STRING, encabezado STRING,
  n_lineas INT, fila_totales INT, tc STRING, fuera_tabla STRING, listas STRING, leyenda STRING, fecha_proceso TIMESTAMP)
USING DELTA COMMENT 'Ppto27 bronze: cada archivo recibido, con hash e identificación (historial de versiones)'
""")

cols = ',\n  '.join(f'{c} STRING' for c in L.COLUMNAS)
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {L.tabla('brz_lineas')} (
  carga_id STRING, sha256 STRING, id_plantilla STRING, ruta_relativa STRING, fila INT,
  {cols},
  tipos STRING, formulas STRING, fecha_proceso TIMESTAMP)
USING DELTA COMMENT 'Ppto27 bronze: filas de Tabla1 sin interpretar (valor como texto, tipo de celda y fórmula)'
""")
# si la tabla se creó con una versión anterior, agregar las columnas nuevas
if 'leyenda' not in spark.table(L.tabla('brz_archivos')).columns:
    spark.sql(f"ALTER TABLE {L.tabla('brz_archivos')} ADD COLUMNS (leyenda STRING)")
display(spark.sql(f"SHOW TABLES IN {L.CATALOGO_UC}.{L.ESQUEMA} LIKE 'ppto27_*'"))